# Занятие 11. Сверточная сеть: от локального окна к классификации картинки

Для самостоятельного разбора PyTorch: [Dive into Deep Learning](https://en.d2l.ai/chapter_preface/index.html).

Полносвязный слой использует отдельный вес для каждого пикселя. Если объект сдвинуть, он попадет на другие веса. Сверточный слой ищет один и тот же локальный признак в разных местах изображения.

Разберем свертку как скользящее локальное окно, затем соберем из слоев CNN. После этого рассмотрим, что меняют нормализация, оптимизатор и обучающие изображения.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
from torch import nn
from IPython.display import display
from support.week11_utils import load_data, show_images, show_table

plt.rcParams.update({"figure.dpi": 110, "axes.spines.top": False, "axes.spines.right": False})

from support.lesson11_visuals import setup_style
setup_style()

In [ ]:
from support.week11_utils import draw_matrices, draw_pipeline, plot_metric_panels

torch.set_num_threads(4)

## 1. Одна свертка: окно, произведения, сумма

Пусть $X$ — изображение с одним каналом размера $H\times W$, $K$ — ядро $k\times k$,
$b$ — одно смещение. При шаге 1 без дополнения нулями выход в положении $(i,j)$:

$$Y_{ij}=b+\sum_{u=0}^{k-1}\sum_{v=0}^{k-1}K_{uv}X_{i+u,j+v}.$$

Сначала выбирают окно $X[i:i+k,j:j+k]$, затем умножают его поэлементно на $K$,
затем складывают произведения и прибавляют $b$. Для ядра
$K=\begin{pmatrix}1&0\\0&-1\end{pmatrix}$ и первого окна
$\begin{pmatrix}1&2\\5&6\end{pmatrix}$ получают $1\cdot1+0\cdot2+0\cdot5-1\cdot6=-5$.

В `torch.nn.Conv2d` ядро не переворачивают: строгое название операции — взаимная
корреляция. В нейросетях ее принято называть сверткой. Обучаемые ядра не обязаны
совпадать с фильтрами обработки изображений.

Матричная запись свертки, im2col и вывод градиентов по входу и ядру приведены в [дополнительном конспекте](notes_11_convolution_matrix.ipynb).

In [ ]:
X = np.arange(1, 17, dtype=float).reshape(4, 4)
K = np.array([[1., 0.], [0., -1.]])
Y = np.zeros((3, 3))
for row in range(3):
    for column in range(3):
        window = X[row:row + 2, column:column + 2]
        products = window * K
        Y[row, column] = products.sum()
print("Вход:\n", X, "\nЯдро:\n", K, "\nВыход:\n", Y)

In [ ]:
from ipywidgets import interact, IntSlider, fixed
from support.lesson11_visuals import show_convolution_window

window_slider = IntSlider(min=0, max=Y.size - 1, value=0, description="окно")
interact(show_convolution_window, X=fixed(X), K=fixed(K), Y=fixed(Y),
         position=window_slider);

## 2. Каналы, padding, stride и dilation

Вход RGB имеет три канала. Одно ядро сверточного слоя содержит по плоскости на каждый входной канал. Отфильтрованные каналы **суммируются** в одну выходную карту. Число выходных каналов задают числом разных ядер $C_{out}$; три входных канала не означают три независимых выходных изображения.

В PyTorch порядок осей — NCHW: батч, каналы, высота, ширина.

### Три входных канала и их сумма

Вычисляют вклады RGB раздельно и проверяют их сумму готовой сверткой.

In [ ]:
demo_data = load_data(train_size=1000, val_size=1000, test_size=100)
rgb_example = demo_data["test_images"][0]
channel_titles = ["R", "G", "B"]
figure, axes = plt.subplots(1, 4, figsize=(12, 3))
axes[0].imshow(rgb_example)
axes[0].set_title("RGB: один объект, три канала")
for channel, axis in enumerate(axes[1:]):
    plane = np.zeros_like(rgb_example)
    plane[:, :, channel] = rgb_example[:, :, channel]
    axis.imshow(plane)
    axis.set_title(channel_titles[channel])
for axis in axes:
    axis.axis("off")
figure.tight_layout()
plt.show()

# Одно выходное ядро суммирует три отфильтрованных входных канала.
input_rgb = torch.tensor(rgb_example.copy()).permute(2, 0, 1).float()[None] / 255
edge_kernel = torch.tensor([[-1., 0., 1.], [-2., 0., 2.], [-1., 0., 1.]])
channel_weights = torch.tensor([1., 0.5, -0.5])
channel_contributions = []
for channel in range(3):
    plane = input_rgb[:, channel:channel + 1]
    weight = edge_kernel[None, None] * channel_weights[channel]
    contribution = nn.functional.conv2d(plane, weight, padding=1)
    channel_contributions.append(contribution[0, 0].numpy())
summed_map = np.stack(channel_contributions).sum(axis=0)
weights_rgb = edge_kernel[None, None] * channel_weights[None, :, None, None]
reference_map = nn.functional.conv2d(input_rgb, weights_rgb, padding=1)[0, 0].numpy()
assert np.allclose(summed_map, reference_map, atol=1e-6)
draw_matrices(channel_contributions + [summed_map],
              ["Вклад R", "Вклад G", "Вклад B", "Сумма: один выходной канал"], annotate=False)

### Обычная свертка: окно перемещается на один пиксель

На анимации вход 4×4, ядро 3×3. Без дополнения ядро помещается в двух положениях по каждой оси: выход 2×2. Одно положение окна дает одно число выхода.

<img src="assets/convolution_arithmetic/no_padding_no_strides.gif" width="300" alt="Вход 4×4, ядро 3×3, шаг 1, без padding; выход 2×2">

Анимации: [Vincent Dumoulin, Francesco Visin — Convolution Arithmetic](https://github.com/vdumoulin/conv_arithmetic).

### Padding: рамка вокруг входа

`padding=1` добавляет по одной строке или столбцу с каждой стороны. При нулевом дополнении это нули: новых обучаемых параметров нет. Ядро теперь может выходить за границы исходной картинки, а выход содержит больше положений.

Здесь вход 5×5, ядро 3×3, padding=1 и stride=1. С учетом рамки доступно 7×7 позиций входа, поэтому выход имеет размер $7-3+1=5$. Для нечетного ядра 3×3 это сохраняет стороны картинки.

<img src="assets/convolution_arithmetic/same_padding_no_strides.gif" width="340" alt="Вход 5×5, рамка padding=1, ядро 3×3, шаг 1; выход 5×5">

Значение padding=1 означает одну клетку **с каждой стороны**, а не одну на всю ось.

### Stride: насколько сдвигают все окно

`stride=2` перемещает начало окна сразу на два пикселя. Точек выхода становится меньше; соседние выходы соответствуют более далеким положениям входа. Расстояние между весами **внутри** ядра при этом не меняется.

В обоих случаях ниже вход 5×5, ядро 3×3 и padding=1. Меняется только stride.

<table><tr><th>stride=1 → выход 5×5</th><th>stride=2 → выход 3×3</th></tr>
<tr><td><img src="assets/convolution_arithmetic/same_padding_no_strides.gif" width="310" alt="Шаг окна равен одному пикселю"></td>
<td><img src="assets/convolution_arithmetic/padding_strides.gif" width="310" alt="Шаг окна равен двум пикселям"></td></tr></table>

Для второго случая число положений по оси равно $\lfloor(5+2-3)/2\rfloor+1=3$. Ядро остается 3×3: stride не уменьшает число его весов.

### Dilation: расстояние между точками внутри ядра

`dilation=2` раздвигает соседние элементы ядра: между ними пропускается один пиксель входа. Девять весов ядра 3×3 охватывают область 5×5, но умножают только девять выбранных пикселей. Дополнительных весов в промежутках нет.

На анимации вход 7×7, padding=0, stride=1 и dilation=2. Само окно по-прежнему перемещается **на один** пиксель, хотя точки внутри него стоят через один. Размер охватываемой области равен $1+(3-1)\cdot2=5$, поэтому выход $7-5+1=3$: карта 3×3.

<img src="assets/convolution_arithmetic/dilation.gif" width="340" alt="Разреженное ядро 3×3 с dilation=2 охватывает 5×5 на входе 7×7">

Stride меняет расстояние между положениями окна; dilation — между отсчетами внутри одного положения.

### Общая формула и проверка в PyTorch

Пусть $H$ — высота входа, $k$ — число весов вдоль стороны ядра, $p$ — padding с каждой стороны, $s$ — stride, $d$ — dilation. По ширине расчет такой же.

У $k$ точек есть $k-1$ промежутков длины $d$. Поэтому охватываемый размер:

$$k_{eff}=1+d(k-1).$$

После дополнения доступна высота $H+2p$. Начало последнего допустимого окна удовлетворяет $is+k_{eff}\le H+2p$. Следовательно,

$$i\le\frac{H+2p-k_{eff}}s,\qquad
H_{out}=\left\lfloor\frac{H+2p-d(k-1)-1}s\right\rfloor+1.$$

Единица появляется потому, что нулевое положение окна тоже дает выход. Округление вниз исключает неполное последнее окно.

Для одного выходного канала с учетом RGB формула значения остается взвешенной суммой:

$$Y_{oij}=b_o+\sum_{c=0}^{C_{in}-1}\sum_{u,v}K_{ocuv}X_{c,\,is-p+ud,\,js-p+vd},$$

где за границами входа стоят нули. Проверим **те же четыре случая**, которые показаны в анимациях.

In [ ]:
geometry_cases = [
    ("Без рамки", 4, 0, 1, 1),
    ("Padding", 5, 1, 1, 1),
    ("Stride", 5, 1, 2, 1),
    ("Dilation", 7, 0, 1, 2),
]
geometry_rows = []
for name, height, padding, stride, dilation in geometry_cases:
    layer = nn.Conv2d(1, 1, kernel_size=3, padding=padding, stride=stride, dilation=dilation)
    output = layer(torch.zeros(1, 1, height, height))
    expected = (height + 2 * padding - dilation * (3 - 1) - 1) // stride + 1
    assert output.shape[-2:] == (expected, expected)
    geometry_rows.append({"случай": name, "вход": f"{height}x{height}",
                          "k": 3, "padding": padding, "stride": stride, "dilation": dilation,
                          "выход": f"{expected}x{expected}"})
show_table(pd.DataFrame(geometry_rows).set_index("случай"))

## 3. Из каких слоев состоит сверточная сеть

| Слой | Действие | Обучаемые параметры |
|:--|:--|:--|
| `Conv2d` | локальная взвешенная сумма по пространству и каналам | ядра и смещения |
| `ReLU` | $\max(0,x)$ поэлементно | нет |
| `MaxPool2d(2)` | максимум в каждом окне $2\times2$, шаг 2 | нет |
| `AvgPool2d(2)` | среднее в каждом окне | нет |
| `AdaptiveAvgPool2d((4,4))` | средние по областям до заданного размера | нет |
| `Flatten` | объединение осей, порядок чисел сохраняется | нет |
| `Linear` | полносвязный классификатор | матрица и смещения |
| `BatchNorm2d` | нормализация каждого канала | масштаб и сдвиг |
| `Dropout` | случайное обнуление при обучении | нет |

После сверточных блоков используют либо `Flatten` и полносвязную часть, либо
глобальное усреднение по пространству (`AdaptiveAvgPool2d(1)`) и малый классификатор.
Во втором случае параметров меньше, но пространственные координаты агрегируются сильнее.
В семинаре сначала собирают CNN с Flatten, затем модель из блоков с усреднением до $4\times4$ перед настраиваемой головой.

Финальный `Linear` выдает десять **оценок**, а не вероятности.
`CrossEntropyLoss` уже включает log-softmax; softmax перед ним не добавляют.
Для выдачи вероятностей при оценке применяют softmax отдельно.

In [ ]:
# Действие каждого слоя проверяют на одних и тех же числах.
layer_input = torch.tensor([[-3., 1., 2., 0.], [4., -2., 1., 3.],
                            [1., 5., -1., 2.], [0., 2., 6., -4.]])[None, None]
relu_map = nn.ReLU()(layer_input)
max_map = nn.MaxPool2d(2)(relu_map)
avg_map = nn.AvgPool2d(2)(relu_map)
global_map = nn.AdaptiveAvgPool2d(1)(relu_map)
draw_matrices([layer_input[0, 0].numpy(), relu_map[0, 0].numpy(),
               max_map[0, 0].numpy(), avg_map[0, 0].numpy(), global_map[0, 0].numpy()],
              ["Вход", "ReLU: отрицательные -> 0", "MaxPool 2x2", "AvgPool 2x2", "Global average"],
              cmap="coolwarm", value_limits=(-4, 8))

### Linear: W x или X W?

У `nn.Linear(4,3)` четыре входных признака и три выходных. PyTorch хранит веса как **W формы 3×4**: строка j содержит четыре веса, которыми вычисляют выход j.

Для одного объекта, записанного столбцом $x\in\mathbb R^{4\times1}$:

$$y_j=\sum_{i=1}^{4}W_{j,i}x_i+b_j,\qquad
\underbrace{y}_{3\times1}=\underbrace{W}_{3\times4}\underbrace{x}_{4\times1}+\underbrace{b}_{3\times1}.$$

В тензоре PyTorch объекты батча лежат **строками**: $X\in\mathbb R^{N\times4}$. Поэтому тот же расчёт записывается иначе:

$$\underbrace{Y}_{N\times3}=\underbrace{X}_{N\times4}\underbrace{W^T}_{4\times3}+\underbrace{b}_{3}.$$

Строка $Y_n=(Wx_n)^T=x_n^TW^T$: транспонирование переводит запись со столбцами в запись со строками. Смещение прибавляется к каждой строке. В примере ниже `bias=False`, поэтому для первого объекта получаем [4, 8, 6] без добавления b.

Ниже настоящий батч из трёх карт 2x2: форма 3x2x2. Flatten сохраняет ось N и даёт X формы 3x4; результат Linear имеет форму 3x3. Цвет рамки связывает карту одного объекта с его строками X и Y. Явную единичную ось канала здесь опускаем: с ней вход имел бы форму 3x1x2x2, а результат Flatten остался бы тем же.

На первом рисунке показана **Wᵀ**, потому что вход изображён строкой. На втором — хранимая **W** и входной столбец. Это два представления одной операции. Запись `X @ W` с W формы 3×4 для четырёх входных признаков не согласована по размерам. [Контракт nn.Linear](https://docs.pytorch.org/docs/2.15/generated/torch.nn.Linear.html).

In [ ]:
from support.lesson11_visuals import draw_array_objects, draw_batched_linear

batch_maps = torch.stack([
    max_map[0, 0],
    torch.tensor([[1., 2.], [3., 4.]]),
    torch.tensor([[2., -1.], [4., 0.]]),
])
flat_map = nn.Flatten(start_dim=1)(batch_maps)
classifier = nn.Linear(4, 3, bias=False)
with torch.no_grad():
    classifier.weight.copy_(torch.tensor([[1., 0., 0., 0.], [0., 1., 1., 0.], [0., 0., 0., 1.]]))

row_scores = flat_map @ classifier.weight.T
column_input = flat_map[0].reshape(4, 1)
column_scores = classifier.weight @ column_input
assert classifier.weight.shape == (3, 4)
assert classifier.weight.T.shape == (4, 3)
assert torch.allclose(classifier(flat_map), row_scores)
assert torch.allclose(column_scores.T, row_scores[:1])
assert batch_maps.shape == (3, 2, 2)
assert flat_map.shape == (3, 4)
assert row_scores.shape == (3, 3)

draw_batched_linear(batch_maps.numpy(), flat_map.numpy(), classifier.weight.T.detach().numpy(),
                    row_scores.detach().numpy());
draw_array_objects([classifier.weight.detach().numpy(), column_input.numpy(), column_scores.detach().numpy()],
                   ["Хранимые веса W", "Столбец x", "Столбец y"],
                   operators=["x", "="], value_limits=(-4, 8),
                   heading="Один объект столбцом: y = W x");

In [ ]:
draw_pipeline(["RGB\nNx3x32x32", "Conv + ReLU\nNx16x32x32", "MaxPool\nNx16x16x16",
               "Conv + ReLU\nNx32x16x16", "Усреднение\nNx32x1x1", "Flatten\nNx32", "Linear\nNx10"],
              "Пример структуры CNN  |  N - размер батча")

### Pooling: что остаётся от окна

Pooling заменяет локальное окно карты признаков одним числом. Для каждого объекта и канала отдельно:

$$Y^{max}_{n,c,i,j}=\max_{0\le u,v<k}X_{n,c,is+u,js+v},\qquad
Y^{avg}_{n,c,i,j}=\frac1{k^2}\sum_{u,v}X_{n,c,is+u,js+v}.$$

В окне $\begin{pmatrix}1&4\\3&2\end{pmatrix}$ MaxPool возвращает 4, AvgPool — (1+4+3+2)/4=2.5. При k=2, stride=2 высота и ширина сокращаются примерно вдвое; N и C сохраняются. Без padding $H_{out}=\lfloor(H-k)/s\rfloor+1$. У слоя нет обучаемых весов; при train и eval вычисления одинаковые.

MaxPool сохраняет самый сильный отклик; AvgPool усредняет все значения окна. Pooling теряет точное положение внутри окна, но не гарантирует устойчивости к любому сдвигу: объект может пересечь границу соседнего окна. [Численные примеры в D2L](https://en.d2l.ai/chapter_convolutional-neural-networks/pooling.html).

In [ ]:
from support.lesson11_visuals import plot_pooling

pooling_input = torch.tensor([[[[1., 4., 2., 0.], [3., 2., 5., 1.],
                                [0., 2., 1., 6.], [7., 3., 4., 2.]]]], requires_grad=True)
maximum_pool = nn.MaxPool2d(2, stride=2)
average_pool = nn.AvgPool2d(2, stride=2)
maximum_values = maximum_pool(pooling_input)
average_values = average_pool(pooling_input)
maximum_gradient = torch.autograd.grad(maximum_values.sum(), pooling_input, retain_graph=True)[0]
average_gradient = torch.autograd.grad(average_values.sum(), pooling_input)[0]

assert maximum_values.shape == average_values.shape == (1, 1, 2, 2)
assert maximum_values[0, 0, 0, 0] == 4
assert average_values[0, 0, 0, 0] == 2.5
maximum_pool.eval()
assert torch.equal(maximum_pool(pooling_input), maximum_values)
plot_pooling(pooling_input.detach().numpy()[0, 0], maximum_values.detach().numpy()[0, 0],
             average_values.detach().numpy()[0, 0], maximum_gradient.numpy()[0, 0],
             average_gradient.numpy()[0, 0]);

### Градиент и перекрытие окон

У MaxPool градиент выхода получает позиция выбранного максимума: остальные позиции данного окна дают ноль. При равенстве максимумов PyTorch выбирает первый по строкам. У AvgPool входящий градиент делится поровну на $k^2$ элементов. Если окна перекрываются, вклады в один входной пиксель **складываются**.

Для центрального максимума 9 ниже все четыре окна выбирают один элемент. При потере $L=\sum Y$ его градиент равен 4, а не 1.

In [ ]:
overlap_input = torch.tensor([[[[1., 2., 0.], [3., 9., 4.], [0., 5., 2.]]]], requires_grad=True)
overlap_output = nn.MaxPool2d(2, stride=1)(overlap_input)
overlap_output.sum().backward()
assert overlap_input.grad[0, 0, 1, 1] == 4
draw_matrices([overlap_input.detach().numpy()[0, 0], overlap_output.detach().numpy()[0, 0],
               overlap_input.grad.numpy()[0, 0]],
              ["Вход", "Выход: четыре максимума 9", "Градиент по входу"])

### Глобальное усреднение

Global average pooling усредняет **всё H×W каждого канала отдельно**:

$$z_{n,c}=\frac1{HW}\sum_{h=0}^{H-1}\sum_{w=0}^{W-1}X_{n,c,h,w}.$$

Размер N×C×H×W превращается в N×C×1×1. В PyTorch это `AdaptiveAvgPool2d((1,1))`; после Flatten остаётся N×C.

In [ ]:
global_average = nn.AdaptiveAvgPool2d((1, 1))(pooling_input)
manual_global_average = pooling_input.mean(dim=(2, 3), keepdim=True)
assert torch.allclose(global_average, manual_global_average)
print("До:", tuple(pooling_input.shape), "-> global average:", tuple(global_average.shape))
print("Одно число на канал:", global_average.detach().flatten().tolist())

### Зачем фиксировать размер перед классификатором

Свёртки принимают изображения разных размеров, но после Flatten линейному слою нужно фиксированное число входных признаков M. Поэтому изображение заранее приводят к одному размеру либо усредняют карты признаков до заданной формы перед Linear. Для выхода 4x4 получаем M=16C, для global average - M=C.

### Как AdaptiveAvgPool2d строит области

Задаётся **размер выхода**, а не размер ядра. Пусть вход имеет высоту H, выход — P. Для выходной строки i, 0≤i<P, PyTorch вычисляет границы входной области:

$$s_i=\left\lfloor\frac{iH}{P}\right\rfloor,\qquad
e_i=\left\lceil\frac{(i+1)H}{P}\right\rceil.$$

Начало включено, конец не включён: используются строки `s_i:e_i`. Аналогично для ширины W и выходной ширины Q:

$$t_j=\left\lfloor\frac{jW}{Q}\right\rfloor,\qquad
u_j=\left\lceil\frac{(j+1)W}{Q}\right\rceil.$$

Затем считают среднее по полученному прямоугольнику, отдельно для n и c:

$$Y_{n,c,i,j}=\frac{1}{(e_i-s_i)(u_j-t_j)}
\sum_{h=s_i}^{e_i-1}\sum_{w=t_j}^{u_j-1}X_{n,c,h,w}.$$

Слой не обучает границы и не решает оптимизационную задачу для kernel_size/stride: границы получаются из этих формул. Они приведены в [реализации PyTorch, start_index/end_index](https://github.com/pytorch/pytorch/blob/main/aten/src/ATen/native/AdaptivePooling.h).

In [ ]:
adaptive_input = torch.arange(40., dtype=torch.float64).reshape(1, 1, 8, 5)
adaptive_output = nn.AdaptiveAvgPool2d((3, 2))(adaptive_input)
manual_adaptive = torch.empty_like(adaptive_output)
for row in range(3):
    start_h = row * 8 // 3
    end_h = ((row + 1) * 8 + 3 - 1) // 3
    for column in range(2):
        start_w = column * 5 // 2
        end_w = ((column + 1) * 5 + 2 - 1) // 2
        region = adaptive_input[:, :, start_h:end_h, start_w:end_w]
        manual_adaptive[:, :, row, column] = region.mean(dim=(2, 3))
assert torch.allclose(adaptive_output, manual_adaptive)
from support.lesson11_visuals import plot_adaptive_regions
plot_adaptive_regions(adaptive_input.numpy()[0, 0], adaptive_output.numpy()[0, 0]);

Дополнительные примеры и замеры: [конспект о слоях](notes_11_layers.ipynb).

## 4. Поле восприятия и перенос признаков

Поле восприятия — область исходного изображения, которая может влиять на один
элемент карты признаков. Пусть $r$ — ее размер, $j$ — расстояние между соседними
элементами карты в координатах входа. Изначально $r_0=1,j_0=1$.

Слой с ядром $k$, dilation $d$ объединяет $d(k-1)+1$ положений. К прежнему полю
добавляется $d(k-1)$ промежутков длины $j$:

$$r_{new}=r+d(k-1)j,\qquad j_{new}=sj.$$

Два слоя $3\times3$ с шагом 1 дают $r_1=3$, $r_2=5$. Между ними стоит нелинейность,
поэтому это не один линейный фильтр $5\times5$.
Свертка с шагом 1 согласована с переносом входа вдали от границ: переносится и карта
признаков. Это **эквивариантность**, а не неизменность ответа. Границы, шаг больше 1,
pooling и полносвязная часть могут нарушить точное равенство. Устойчивость к переносу
не гарантируется одним наличием сверточных слоев.

In [ ]:
# В трех слоях 3x3 без pooling центральный выход зависит от 3x3, 5x5, 7x7 входа.
receptive_masks = []
for radius in [1, 2, 3]:
    mask = np.zeros((9, 9))
    mask[4 - radius:5 + radius, 4 - radius:5 + radius] = 1
    mask[4, 4] = 2
    receptive_masks.append(mask)
draw_matrices(receptive_masks, ["Один Conv: 3x3", "Два Conv: 5x5", "Три Conv: 7x7"], cmap="Blues")

# Перенос без циклического оборачивания, нулевая граница; отступ исключают из проверки.
shift_input = torch.zeros(1, 1, 12, 12)
shift_input[0, 0, 4:7, 4:7] = 1
shifted_input = torch.zeros_like(shift_input)
shifted_input[:, :, 1:, 1:] = shift_input[:, :, :-1, :-1]
fixed_kernel = torch.tensor([[1., 0., -1.], [2., 0., -2.], [1., 0., -1.]])[None, None]
first_map = nn.functional.conv2d(shift_input, fixed_kernel, padding=1)
shifted_map = nn.functional.conv2d(shifted_input, fixed_kernel, padding=1)
assert torch.equal(shifted_map[:, :, 2:-1, 2:-1], first_map[:, :, 1:-2, 1:-2])
draw_matrices([shift_input[0, 0].numpy(), first_map[0, 0].numpy(),
               shifted_input[0, 0].numpy(), shifted_map[0, 0].numpy()],
              ["Объект", "Карта признаков", "Объект сдвинут", "Карта тоже сдвинута"], annotate=False)

## 5. Параметры и вычисления при разных размерах изображения

MAC (multiply–accumulate) — умножение с прибавлением результата к накопленной сумме. Один вес может участвовать во многих MAC. Для `Linear(n_in,n_out)`:

$$P_{linear}=(n_{in}+1)n_{out},\qquad MAC_{linear}=n_{in}n_{out}.$$

Для `Conv2d` с groups=1:

$$P_{conv}=(C_{in}k_hk_w+1)C_{out},\qquad
MAC_{conv}=H_{out}W_{out}C_{out}C_{in}k_hk_w.$$

Число параметров относится к **модели** и не умножается на число картинок. MAC считаем на один forward одной картинки.

Сравним две сети с одинаковым числом выходов — 10 — для RGB-входа S×S:

- **MLP:** Flatten → Linear(3S²,128) → ReLU → Linear(128,10). При смене S меняется только вход первого Linear. Ширина 128 и последний слой остаются прежними;
- **CNN:** Conv2d(3,16,3,padding=1) → ReLU → global average pooling → Flatten → Linear(16,10). Все веса остаются той же формы при любом допустимом S.

Для MLP суммируем параметры и MAC двух линейных слоёв:

$$P_{MLP}=(3S^2+1)128+(128+1)10=384S^2+1418,$$
$$MAC_{MLP}=3S^2\cdot128+128\cdot10=384S^2+1280.$$

Для CNN:

$$P_{CNN}=(3\cdot3\cdot3+1)16+(16+1)10=618,$$
$$MAC_{CNN}=S^2\cdot16\cdot3\cdot3\cdot3+16\cdot10=432S^2+160.$$

Параметры CNN дают горизонтальную линию; её вычисления растут с площадью изображения. У MLP с фиксированной шириной головы обе величины тоже растут как S². Для каждого S рассматривается отдельная MLP с первым слоем нужной формы: обученные веса этого слоя нельзя автоматически перенести с одного S на другой.

В MAC включены Conv и Linear. Усреднение, ReLU, перемещения памяти и накладные расходы не переведены в MAC; это не оценка секунд. Поэтому даже при большом выигрыше по числу параметров CNN здесь не обязана выигрывать по MAC или задержке. Время измеряют отдельно.

In [ ]:
from support.week11_utils import parameter_count
from support.lesson11_visuals import plot_size_cost

# У CNN один и тот же объект модели для всех размеров входа.
cost_cnn = nn.Sequential(nn.Conv2d(3, 16, 3, padding=1), nn.ReLU(),
                         nn.AdaptiveAvgPool2d(1), nn.Flatten(), nn.Linear(16, 10))
cost_rows = []
for side in [8, 16, 24, 32, 48, 64, 96, 128]:
    cost_mlp = nn.Sequential(nn.Flatten(), nn.Linear(3 * side * side, 128), nn.ReLU(), nn.Linear(128, 10))
    mlp_parameters = 384 * side * side + 1418
    cnn_parameters = 618
    assert parameter_count(cost_mlp) == mlp_parameters
    assert parameter_count(cost_cnn) == cnn_parameters
    example = torch.zeros(2, 3, side, side)
    assert cost_mlp(example).shape == cost_cnn(example).shape == (2, 10)
    cost_rows.append({"сторона S": side, "MLP параметры": mlp_parameters, "CNN параметры": cnn_parameters,
                      "MLP MAC": 384 * side * side + 1280, "CNN MAC": 432 * side * side + 160})
cost_table = pd.DataFrame(cost_rows)
plot_size_cost(cost_table);

## 6. Зачем нормализовать признаки и активации

Начнём с линейного классификатора двух классов. Один объект описывается признаками $x_1,x_2$. Его **оценка z до sigmoid** вычисляется так:

$$z=w_1x_1+w_2x_2+b=w^Tx+b.$$

Здесь $w_1,w_2$ — веса, b — смещение. Вероятность класса 1 равна $p=1/(1+e^{-z})$; класс 1 выбирают при z>0. Разделяющая прямая задаётся уравнением z=0. Loss ниже — средняя бинарная кросс-энтропия по 12 показанным объектам.

Если поменять веса на $\Delta w$, не меняя объект, то

$$z_{new}=(w+\Delta w)^Tx+b=z+\Delta w^Tx,$$
$$\Delta z=\Delta w^Tx.$$

Поэтому изменение одного веса на 0.25 может заметно сдвинуть оценки, когда соответствующий признак имеет большие значения. Чтобы показать этот эффект, выразим **один и тот же классификатор** в исходных и нормированных координатах. Схема двух классов и границы используется в [CS231n: Data preprocessing](https://cs231n.github.io/neural-networks-2/).

### От стандартизации к пересчёту весов

Для каждого признака j обозначим через $\mu_j$ его среднее, через $\sigma_j>0$ — стандартное отклонение. Стандартизация вычитает среднее и делит на стандартное отклонение:

$$x'_j=\frac{x_j-\mu_j}{\sigma_j}.$$

Отсюда получаем обратную запись в два шага:

$$\sigma_jx'_j=x_j-\mu_j,\qquad x_j=\sigma_jx'_j+\mu_j.$$

Это преобразование координат, а не предположение о нормальном распределении данных. В этом учебном примере статистики считаются по 12 точкам; в задаче с разбиением статистики предобработки считают по train.

Теперь подставим выражение для x в **исходную оценку классификатора**:

$$z=\sum_jw_jx_j+b
=\sum_jw_j(\sigma_jx'_j+\mu_j)+b.$$

Раскрываем скобки и отдельно собираем слагаемые с $x'_j$ и константы:

$$z=\sum_j(\sigma_jw_j)x'_j+\left(b+\sum_jw_j\mu_j\right).$$

Значит, для нормированных признаков нужно взять веса и смещение

$$w'_j=\sigma_jw_j,\qquad b'=b+\sum_jw_j\mu_j.$$

Тогда для **каждого того же объекта** оценка $w'^Tx'+b'$ равна прежней $w^Tx+b$. Совпадают и вероятности sigmoid, и исходный loss. Численные значения весов при этом изменились: они записаны в другой системе координат.

In [ ]:
# Синтетические данные; классы отмечены кругами и треугольниками.
norm_red = np.array([[-1.2, -.2], [-.9, .9], [-1.3, -.8],
                     [-.7, .3], [-1., -.7], [-.5, 1.]])
norm_points = np.vstack([norm_red, -norm_red])
norm_points = norm_points * np.array([.7, 2.2]) + np.array([.8, -8.])
norm_labels = np.r_[np.zeros(6), np.ones(6)]
norm_mean = norm_points.mean(axis=0)
norm_std = norm_points.std(axis=0)
norm_scaled = (norm_points - norm_mean) / norm_std

# w' = sigma * w, b' = b + w^T mu сохраняют исходные оценки.
norm_weight = np.array([4., .4])
norm_scaled_weight = norm_std * norm_weight
norm_scaled_bias = float(norm_mean @ norm_weight)
norm_original_scores = norm_points @ norm_weight
norm_scaled_scores = norm_scaled @ norm_scaled_weight + norm_scaled_bias
assert np.allclose(norm_original_scores, norm_scaled_scores)
norm_original_probabilities = 1 / (1 + np.exp(-norm_original_scores))
norm_scaled_probabilities = 1 / (1 + np.exp(-norm_scaled_scores))
assert np.allclose(norm_original_probabilities, norm_scaled_probabilities)

norm_delta = np.array([0., .25])
figure, axes = plt.subplots(1, 2, figsize=(13, 6))
norm_panels = [
    (norm_points, norm_weight, 0., "Исходные признаки x", (-2.5, 4.5), (-11, 2)),
    (norm_scaled, norm_scaled_weight, norm_scaled_bias,
     "Нормированные признаки x_prime", (-2.5, 2.5), (-2.5, 3.1)),
]
norm_sensitivity_rows = []
for axis, (points, weight, bias, title, x_limits, y_limits) in zip(axes, norm_panels):
    axis.axhline(0, color="#a0a7af", linewidth=1)
    axis.axvline(0, color="#a0a7af", linewidth=1)
    axis.scatter(points[:6, 0], points[:6, 1], color="#d94e46", s=95, label="Класс 0", zorder=3)
    axis.scatter(points[6:, 0], points[6:, 1], color="#2446b3", marker="^", s=120,
                 label="Класс 1", zorder=3)

    line_y = np.linspace(*y_limits, 100)
    line_x = -(weight[1] * line_y + bias) / weight[0]
    changed_weight = weight + norm_delta
    changed_x = -(changed_weight[1] * line_y + bias) / changed_weight[0]
    axis.plot(line_x, line_y, color="#1c232b", linewidth=2.5, label="До изменения веса")
    axis.plot(changed_x, line_y, color="#d18f2f", linewidth=2, linestyle="--", label="После +0.25 ко второму весу")

    scores = points @ weight + bias
    changed_scores = points @ changed_weight + bias
    loss = np.mean(np.logaddexp(0, scores) - norm_labels * scores)
    changed_loss = np.mean(np.logaddexp(0, changed_scores) - norm_labels * changed_scores)
    changed_answers = (scores > 0) != (changed_scores > 0)
    if changed_answers.any():
        axis.scatter(points[changed_answers, 0], points[changed_answers, 1], s=240,
                     facecolors="none", edgecolors="#d18f2f", linewidth=1.5, zorder=4)
    annotation = f"Loss: {loss:.3f} -> {changed_loss:.3f}\nИзменили ответ: {changed_answers.sum()} из 12"
    axis.text(.04, .96, annotation, transform=axis.transAxes, va="top", fontsize=11,
              bbox={"facecolor": "white", "alpha": .9, "edgecolor": "none"})
    axis.set(title=title, xlim=x_limits, ylim=y_limits, xlabel="x_prime1" if title.startswith("Нормированные") else "x1",
             ylabel="x_prime2" if title.startswith("Нормированные") else "x2")
    norm_sensitivity_rows.append({"координаты": title, "loss": loss,
                                  "loss после шага": changed_loss, "ответов изменено": changed_answers.sum()})

handles, labels = axes[0].get_legend_handles_labels()
figure.legend(handles, labels, loc="lower center", ncol=4, frameon=False)
figure.tight_layout(rect=(0, .08, 1, 1))
plt.show()
show_table(pd.DataFrame(norm_sensitivity_rows).set_index("координаты"))
assert np.isclose(norm_sensitivity_rows[0]["loss"], norm_sensitivity_rows[1]["loss"])

### Что сравнивается на двух панелях

**Слева** показаны исходные признаки x, **справа** — их нормированные значения x′. Круг и треугольник обозначают класс; справа каждый объект сохранил свой класс, но сменил координаты.

Сплошные чёрные прямые — исходный классификатор: слева с весами w,b, справа с пересчитанными w′,b′. Они дают одинаковые оценки и вероятности для соответствующих объектов, поэтому **до изменения веса loss равен примерно 0.145 на обеих панелях**. Это и проверяется через `np.allclose`.

Далее отдельно меняем **второй вес на +0.25**: слева w₂, справа w′₂. Оранжевые пунктирные прямые показывают результат. Это заданное вручную изменение веса, а не шаг, вычисленный оптимизатором. Одинаковая численная прибавка в разных координатах не обязана задавать одинаковое изменение функции.

Слева второй признак близок к −8. Поэтому оценка объекта получает сдвиг около $0.25\cdot(-8)=-2$: loss вырастает примерно до 0.404, три объекта меняют ответ. Справа x′₂ центрирован и имеет единичный масштаб; loss меняется примерно до 0.179, ответы остаются прежними. Пример показывает чувствительность к выбранному изменению параметра.

### Почему нормируют значения внутри сети

Для следующего слоя исходными данными служит уже не картинка, а значения h, полученные от предыдущего слоя. Здесь $h_j$ — j-е входное значение этого слоя, $w_{kj}$ — вес связи с его k-м выходом, $b_k$ — смещение. До функции активации этот выход равен

$$z_k=\sum_jw_{kj}h_j+b_k.$$

Пусть L — потеря одного объекта. При изменении $w_{kj}$ вход $h_j$ не меняется, поэтому

$$\frac{\partial z_k}{\partial w_{kj}}=h_j.$$

По правилу цепочки:

$$\frac{\partial L}{\partial w_{kj}}
=\frac{\partial L}{\partial z_k}\frac{\partial z_k}{\partial w_{kj}}
=\frac{\partial L}{\partial z_k}h_j.$$

Так масштаб входных значений h непосредственно входит в градиент по весам. Большие и сильно различающиеся значения могут давать разные масштабы обновлений разных весов. Нормализация исходной картинки не задаёт масштабы всех h: предыдущие слои преобразуют их, а их веса меняются во время обучения.

BatchNorm применяет нормирование к промежуточным значениям сети. Обучаемые масштаб и сдвиг позволяют затем выбрать подходящее представление. Ниже разберём, по каким осям считаются статистики и как меняются вычисления в train и eval.

### Что именно вычисляет BatchNorm

Для одного канала собирают значения $x_1,\ldots,x_M$. У `BatchNorm1d` на входе N×C это $M=N$; у `BatchNorm2d` на входе N×C×H×W — $M=NHW$, то есть батч и обе пространственные оси. Каждый канал обрабатывают отдельно.

**На train:**

$$\mu_{\mathcal B}=\frac1M\sum_{i=1}^M x_i,$$
$$v_{\mathcal B}=\frac1M\sum_{i=1}^M(x_i-\mu_{\mathcal B})^2,$$
$$\hat x_i=\frac{x_i-\mu_{\mathcal B}}{\sqrt{v_{\mathcal B}+\varepsilon}},$$
$$y_i=\gamma\hat x_i+\beta.$$

Вычитание среднего центрирует канал; деление задает масштаб. $\varepsilon$ защищает знаменатель при малом разбросе. $\gamma$ и $\beta$ обучают обратным распространением, по два числа на канал.

### Что сохраняют для inference

В PyTorch по умолчанию `track_running_stats=True`. Кроме параметров $\gamma,\beta$ слой хранит буферы `running_mean` и `running_var`. На train они обновляются экспоненциальным усреднением с коэффициентом $\alpha=\texttt{momentum}$:

$$\mu_{run}\leftarrow(1-\alpha)\mu_{run}+\alpha\mu_{\mathcal B},$$
$$v_{run}\leftarrow(1-\alpha)v_{run}+\alpha s_{\mathcal B}^2,
\qquad s_{\mathcal B}^2=\frac{M}{M-1}v_{\mathcal B}\quad(M>1).$$

Нормирование текущего батча использует дисперсию с $1/M$, а накопление PyTorch — несмещенную оценку с $1/(M-1)$. `momentum` здесь задает долю нового наблюдения; это другой смысл, чем momentum оптимизатора. Начальные буферы равны 0 и 1.

**На inference после `eval()`:**

$$y_i=\gamma\frac{x_i-\mu_{run}}{\sqrt{v_{run}+\varepsilon}}+\beta.$$

Статистики текущего тестового батча не используют, буферы не обновляют. Так ответ одного изображения не зависит от соседей в запросе; можно обрабатывать и один объект. Параметры и накопленные буферы сохраняются в `state_dict`. Режим без накопления статистик — отдельный случай, в следующем примере осей он указан явно.

[Проверенные соглашения PyTorch BatchNorm2d](https://docs.pytorch.org/docs/2.15/generated/torch.nn.BatchNorm2d.html).

In [ ]:
import copy

# Четыре объекта, два признака. Оси BatchNorm1d проще проверить вручную.
bn_input = torch.tensor([[1., 10.], [3., 14.], [5., 18.], [7., 22.]])
bn_demo = nn.BatchNorm1d(2, momentum=.25)
with torch.no_grad():
    bn_demo.weight.copy_(torch.tensor([2., .5]))
    bn_demo.bias.copy_(torch.tensor([.5, -1.]))

batch_mean = bn_input.mean(dim=0)
batch_variance = bn_input.var(dim=0, unbiased=False)
unbiased_variance = bn_input.var(dim=0, unbiased=True)
manual_normalized = (bn_input - batch_mean) / torch.sqrt(batch_variance + bn_demo.eps)
manual_output = bn_demo.weight * manual_normalized + bn_demo.bias

bn_demo.train()
bn_train_output = bn_demo(bn_input)
assert torch.allclose(bn_train_output, manual_output, atol=1e-6)
assert torch.allclose(bn_demo.running_mean, .25 * batch_mean)
assert torch.allclose(bn_demo.running_var, .75 * torch.ones(2) + .25 * unbiased_variance)

bn_statistics = pd.DataFrame({
    "mu батча": batch_mean.numpy(), "v батча (1/M)": batch_variance.numpy(),
    "s^2 батча (1/(M-1))": unbiased_variance.numpy(),
    "running_mean": bn_demo.running_mean.numpy(), "running_var": bn_demo.running_var.numpy(),
}, index=["канал 0", "канал 1"])
show_table(bn_statistics)

bn_demo.eval()
buffers_before_eval = (bn_demo.running_mean.clone(), bn_demo.running_var.clone())
bn_eval_output = bn_demo(bn_input)
manual_eval_output = bn_demo.weight * (bn_input - bn_demo.running_mean)
manual_eval_output = manual_eval_output / torch.sqrt(bn_demo.running_var + bn_demo.eps) + bn_demo.bias
assert torch.allclose(bn_eval_output, manual_eval_output, atol=1e-6)
assert torch.equal(bn_demo.running_mean, buffers_before_eval[0])
assert torch.equal(bn_demo.running_var, buffers_before_eval[1])
print("Формулы train, running-статистики и eval совпали с PyTorch.")

### Меняем состав батча, сохраняя первый объект

В каждом батче четыре объекта и два канала. Строка матрицы - объект, столбец - канал. Первый объект в A и B одинаковый: [1, 10]. У остальных трёх объектов батча B значения увеличены на 20.

На рисунке показаны полные входы и выходы формы 4x2. Золотая рамка выделяет первый объект, канал 0. Для него расписаны средние, дисперсии и подстановки в формулы. Два результата train относятся к двум отдельным запускам.

Для train используются две независимые копии слоя. Для eval оба батча обрабатываются исходным слоем, у которого сохранены статистики после одного прохода A. Их обновление с momentum=0.25 также показано на рисунке.

In [ ]:
# Первый объект одинаковый; остальные три объекта батча B сдвинуты.
bn_other_batch = bn_input.clone()
bn_other_batch[1:] += 20
train_copy_a = copy.deepcopy(bn_demo).train()
train_copy_b = copy.deepcopy(bn_demo).train()
train_output_a = train_copy_a(bn_input).detach()
train_output_b = train_copy_b(bn_other_batch).detach()
eval_output_a = bn_demo(bn_input).detach()
eval_output_b = bn_demo(bn_other_batch).detach()
assert torch.allclose(eval_output_a[0], eval_output_b[0])

from support.lesson11_visuals import plot_bn_batch_effect
plot_bn_batch_effect(
    bn_input.numpy(), bn_other_batch.numpy(),
    np.stack([train_output_a.numpy(), train_output_b.numpy()]),
    np.stack([eval_output_a.numpy(), eval_output_b.numpy()]),
    gamma=bn_demo.weight.detach().numpy(), beta=bn_demo.bias.detach().numpy(),
    running_mean=bn_demo.running_mean.numpy(), running_variance=bn_demo.running_var.numpy(),
    momentum=bn_demo.momentum, eps=bn_demo.eps,
);

### Какие оси нормирует BatchNorm

На схеме **Batch Norm** синим выделены значения, для которых вычисляют одну пару среднего и дисперсии. Ось N — объекты батча, C — каналы, (H,W) — пространственные позиции.

<img src="assets/normalization_dropout/batch_norm_axes.png" width="360" alt="Batch Norm: один канал, все объекты N и пространственные позиции H,W">

Источник: [Wu, He. Group Normalization (2018), рис. 2, Batch Norm](https://arxiv.org/abs/1803.08494).

Синяя плоскость проходит через **все объекты N и все пространственные позиции H,W одного канала C**. Для входа N×C×H×W усреднение идёт по (N,H,W), то есть `dim=(0,2,3)`. Каналы не смешиваются. Каждая статистика имеет форму 1×C×1×1 и распространяется на исходный тензор по правилам broadcasting.

Тензорный пример со свёрткой и поканальными вычислениями: [конспект о слоях](notes_11_layers.ipynb).

## 7. Dropout: train, inference и два соглашения о масштабе

In [ ]:
from support.lesson11_visuals import plot_dropout_network

# Локальный seed не меняет генератор остальных примеров.
with torch.random.fork_rng(devices=[]):
    torch.manual_seed(23)
    dropout_demo = nn.Dropout(p=0.5)
    dropout_demo.train()
    masks = []
    for forward_index in range(2):
        first_hidden = dropout_demo(torch.ones(6)).numpy()
        second_hidden = dropout_demo(torch.ones(5)).numpy()
        masks.append([first_hidden, second_hidden])

assert all(np.isin(layer, [0., 2.]).all() for mask in masks for layer in mask)
assert any(not np.array_equal(left, right) for left, right in zip(masks[0], masks[1]))
plot_dropout_network(masks, probability=0.5);

Крестик означает нулевую активацию; её входящие и исходящие связи не участвуют в этом проходе. Dropout включён на двух скрытых слоях. Два правых рисунка показывают независимые маски, полученные с помощью `nn.Dropout`.

Источник идеи: [Srivastava et al. (2014), рис. 1](https://jmlr.org/papers/v15/srivastava14a.html). [Оригинальная схема](assets/normalization_dropout/dropout_fig1.png).

Пусть $p$ — вероятность **выключения**, $q=1-p$ — вероятность сохранения; $m_i\sim\operatorname{Bernoulli}(q)$. Например, в `nn.Dropout(p=0.2)` сохраняется в среднем 80% активаций.

В классической записи на train оставляют $\tilde h_i=m_i h_i$. Тогда $\mathbb E[\tilde h_i]=q h_i$, поэтому на inference используют $q h_i$. Эквивалентно этот множитель можно учесть в весах следующего линейного слоя. Если в записи буквой $p$ называют вероятность **сохранения**, множитель на inference действительно равен тому $p$.

**PyTorch использует inverted dropout:**

$$\text{train: }\quad\tilde h_i=\frac{m_i h_i}{q},\qquad
\mathbb E[\tilde h_i]=\frac{q h_i}{q}=h_i;$$
$$\text{inference после eval(): }\quad h_i^{eval}=h_i.$$

Масштаб компенсирован уже при обучении: оставшиеся значения увеличиваются в $1/q$ раз. На inference маски нет и дополнительного умножения на $q$ тоже нет. Для $p=0.5$: train дает 0 или $2h_i$, eval возвращает $h_i$. Равенство относится к математическому ожиданию; сумма одного случайного батча не обязана совпасть точно.

Обычный `Dropout` обнуляет элементы, `Dropout2d` — целые каналы. В семинаре Dropout2d стоит после пары сверток с $p=0.2$. [Соглашение PyTorch Dropout](https://docs.pytorch.org/docs/2.15/generated/torch.nn.Dropout.html).

Weight decay уменьшает веса при обновлении. Для градиентного спуска штраф
$L_{total}=L+\lambda\|w\|^2/2$ дает
$\nabla L_{total}=g+\lambda w$, откуда
$w_{new}=(1-\eta\lambda)w-\eta g$.
В адаптивных оптимизаторах добавление $\lambda w$ к градиенту и отдельное уменьшение
веса не эквивалентны. AdamW отделяет уменьшение веса от накопления моментов градиента.

In [ ]:
# Маска и усиление оставшихся активаций видны пространственно.
torch.manual_seed(17)
activation_grid = torch.arange(1, 13, dtype=torch.float32).reshape(3, 4) / 2
mask_layer = nn.Dropout(0.5)
mask_layer.train()
masked_grid = mask_layer(activation_grid)
mask_layer.eval()
evaluation_grid = mask_layer(activation_grid)
draw_matrices([activation_grid.numpy(), masked_grid.numpy(), evaluation_grid.numpy()],
              ["Активации", "train: нули и значения x2", "eval: исходные значения"], cmap="Blues")
assert torch.equal(evaluation_grid, activation_grid)
assert torch.all((masked_grid == 0) | (masked_grid == 2 * activation_grid))

### Как переключать режимы модели

| Слой | `model.train()` | `model.eval()` |
|:--|:--|:--|
| BatchNorm, стандартный режим | нормирует текущим батчем; обновляет running mean/variance | использует сохраненные running mean/variance; не обновляет их |
| Dropout в PyTorch | новая случайная маска; сохраненные значения делятся на $q=1-p$ | маски нет; выход равен входу |

На train BatchNorm получает статистики из доступных батчей, а Dropout создает случайные варианты сети. На inference нужны воспроизводимые ответы с согласованным масштабом: BatchNorm берет накопленные оценки, Dropout возвращает все активации.

`eval()` меняет поведение слоев, но не отключает вычисление градиентов. `no_grad()` и `inference_mode()` отключают запись графа производных, но сами не переключают BatchNorm и Dropout. Поэтому для оценки выполняют **оба** действия: `model.eval()`, затем forward внутри `torch.inference_mode()`. Перед следующим обучением снова вызывают `model.train()`.

## 8. Как оптимизаторы движутся по поверхности потерь

Выберите поверхность, поверните её мышью и включите нужные методы. На каждой поверхности у всех один старт и одинаковое число шагов; настройки видны в раскрывающемся блоке. Формулы, разбор механизмов и память состояний собраны в [конспекте оптимизаторов](notes_11_optimizers.ipynb).

In [ ]:
from support.optimizer_dashboard import build_surface_examples, export_dashboard, display_dashboard

optimizer_examples, optimizer_settings = build_surface_examples()
optimizer_html = export_dashboard(optimizer_examples, path="outputs/optimizer_landscapes_3d.html", trajectory_only=True)
optimizer_button = display_dashboard(optimizer_html, height=1200)

## 9. Аугментация: один объект в разных условиях

Свертка использует общие веса, но сама по себе не гарантирует устойчивость ко всем сдвигам, освещению или шуму. Во время обучения можно давать случайные варианты картинки с той же меткой.

Если $T$ — случайное допустимое преобразование, вместо потери на исходном объекте минимизируют $\mathbb E_T L(f(T(x)),y)$. Один вызов дает одну выборку из этого ожидания. Поворот, сдвиг или отражение допустимы только тогда, когда сохраняют смысл класса.

На CIFAR-10 используют умеренный crop, горизонтальное отражение и изменение яркости. Одну картинку ниже преобразуют несколько раз.

In [ ]:
# Действительные train-аугментации: один объект, новые варианты при каждом чтении.
from torchvision.transforms import v2
raw_augmentation = torch.tensor(demo_data["train_images"][0].copy()).permute(2, 0, 1)
crop_only = v2.RandomCrop(32, padding=4, padding_mode="reflect")
flip_only = v2.RandomHorizontalFlip(p=1.0)
color_only = v2.ColorJitter(brightness=.3, contrast=.3)
augmented_views = [raw_augmentation, crop_only(raw_augmentation), flip_only(raw_augmentation), color_only(raw_augmentation)]
visible_views = [view.permute(1, 2, 0).numpy() for view in augmented_views]
show_images(visible_views, ["Исходная", "Случайный crop", "Горизонтальное отражение", "Яркость и контраст"], columns=4)

combined_augmentation = v2.Compose([crop_only, v2.RandomHorizontalFlip(.5), color_only])
repeated_views = [combined_augmentation(raw_augmentation).permute(1, 2, 0).numpy() for _ in range(6)]
show_images(repeated_views, [f"новый вызов {index + 1}" for index in range(6)])

На каждом новом чтении train получается новый вариант. Validation и test преобразуют детерминированно.

Проверка на практике: обучить две одинаковые CNN, меняя только аугментации train, и подать им одни и те же испорченные test-картинки. Изменение яркости и сдвиг близки к обучающим преобразованиям; устойчивость к шуму из этого автоматически не следует.

При сравнении MLP и CNN учитывают качество по классам, ошибки вероятностей, время обучения, число параметров и скорость forward. Меньшее число весов само по себе не обещает меньшей задержки.

### Дополнительные разборы

- [Dive into Deep Learning: Batch Normalization](https://en.d2l.ai/chapter_convolutional-modern/batch-norm.html): формула, ручная реализация и проверка готовым слоем. В этом ноутбуке тот же учебный приём применён к малому тензору с видимыми осями.
- [CS231n: устройство свёрточных сетей](https://cs231n.github.io/convolutional-networks/): пространственные размеры и каналы на каждом слое.
- [CS231n: просмотр внутренних представлений](https://cs231n.github.io/understanding-cnn/): карты активаций обученной сети. На семинаре строим такие карты для своей CNN.